# DSPy Customer Care Agent

**Agentic chatbot for customer support — built with DSPy + Ollama (local)**

This notebook walks through the full DSPy workflow in one place:

| Section | What it covers |
|---|---|
| 1. Setup | Install DSPy, configure models |
| 2. Signatures | Define the 3 LLM steps |
| 3. Agent | Chain signatures into a pipeline |
| 4. Trainset | 31 labeled examples + train/dev split |
| 5. Metrics | `intent_accuracy` + LLM-judge `combined_metric` |
| 6. Baseline | Run the agent before any optimization |
| 7. Optimize | Run BootstrapFewShot or MIPROv2 (toggle below) |
| 8. Compare | Before vs after accuracy |
| 9. Inference | Load compiled prompt + run live test cases |

---
> **Requirements**: Ollama running locally with `qwen2.5:14b` (student) and `qwen2.5:32b` (teacher) pulled.

## Section 1 — Setup

Install DSPy and configure both LLM models.

- **Student** (`qwen2.5:14b`) — runs in production, used for all inference
- **Teacher** (`qwen2.5:32b`) — only used during MIPROv2 optimization to propose better instructions

In [1]:
# Install DSPy (skip if already installed)
#!pip install dspy-ai -q   
# ran from terminal in .venv

In [2]:
import os
import json
import dspy
from dspy.teleprompt import MIPROv2, BootstrapFewShot

# ── Model settings ────────────────────────────────────────────────────────────
OLLAMA_BASE_URL    = "http://localhost:11434"
STUDENT_MODEL_ID   = "ollama/qwen2.5:14b"   # deployed in production
TEACHER_MODEL_ID   = "ollama/qwen2.5:32b"   # only used during optimization
MAX_TOKENS         = 1000

# ── Optimizer toggle ──────────────────────────────────────────────────────────
# "miprov2"   → best quality — teacher rewrites instructions + Bayesian search
# "bootstrap" → faster      — only selects best few-shot examples
OPTIMIZER          = "miprov2"
MIPRO_AUTO         = "light"     # "light" (~10 trials) | "medium" (~25) | "heavy" (~50+)
BOOTSTRAP_MAX_DEMOS = 4

# ── Save paths ────────────────────────────────────────────────────────────────
os.makedirs("optimized", exist_ok=True)
MIPRO_SAVE_PATH     = "optimized/miprov2_agent.json"
BOOTSTRAP_SAVE_PATH = "optimized/bootstrap_agent.json"
TRAIN_SPLIT         = 0.8

# ── Configure LLMs ───────────────────────────────────────────────────────────
STUDENT_MODEL = dspy.LM(
    model=STUDENT_MODEL_ID,
    api_base=OLLAMA_BASE_URL,
    api_key="ollama",
    max_tokens=MAX_TOKENS,
)

TEACHER_MODEL = dspy.LM(
    model=TEACHER_MODEL_ID,
    api_base=OLLAMA_BASE_URL,
    api_key="ollama",
    max_tokens=MAX_TOKENS,
)

# Student is the default active model
dspy.configure(lm=STUDENT_MODEL)

print(f"Student model : {STUDENT_MODEL_ID}")
print(f"Teacher model : {TEACHER_MODEL_ID}")
print(f"Optimizer     : {OPTIMIZER}")
print(f"Ollama URL    : {OLLAMA_BASE_URL}")

Student model : ollama/qwen2.5:14b
Teacher model : ollama/qwen2.5:32b
Optimizer     : miprov2
Ollama URL    : http://localhost:11434


## Section 2 — Signatures

A **Signature** defines one LLM step as a typed class.

| Part | What you write | What DSPy does with it |
|---|---|---|
| Docstring | Task instruction | Becomes the instruction line in the prompt |
| `InputField` | Field name + desc | Becomes the input label |
| `OutputField` | Field name + desc | Becomes the output slot the LLM fills |

You never write `"You are a helpful agent..."` by hand. DSPy assembles the full prompt from these three parts automatically.

Three signatures for this pipeline:
1. `ClassifyIntent` — what does the customer want?
2. `ExtractEntities` — what specific data did they mention?
3. `GenerateResponse` — what should we say back?

In [3]:
class ClassifyIntent(dspy.Signature):
    """Classify the customer support message into a single known intent."""
    #                ↑ Docstring becomes the Task instruction in the prompt

    user_message: str = dspy.InputField(
        desc="Raw message from the customer"
    )
    intent: str = dspy.OutputField(
        # desc acts as a soft constraint — LLM will stay within these values
        desc="Exactly one of: track_order, cancel_order, refund_request, escalate, general_query"
    )
    confidence: float = dspy.OutputField(
        desc="Confidence score between 0.0 and 1.0"
    )


class ExtractEntities(dspy.Signature):
    """Extract structured entities from the customer message."""

    user_message: str = dspy.InputField(
        desc="Raw message from the customer"
    )
    order_id: str = dspy.OutputField(
        desc="Order ID if mentioned (e.g. #12345), else empty string"
    )
    customer_email: str = dspy.OutputField(
        desc="Customer email if mentioned, else empty string"
    )
    issue_summary: str = dspy.OutputField(
        desc="One sentence summary of the customer's issue"
    )


class GenerateResponse(dspy.Signature):
    """Generate a helpful, polite, and concise customer care response."""

    user_message: str = dspy.InputField(
        desc="Raw message from the customer"
    )
    intent: str = dspy.InputField(
        desc="Classified intent of the message"
    )
    entities: str = dspy.InputField(
        desc="JSON string with keys: order_id, email, issue_summary"
    )
    policy_context: str = dspy.InputField(
        desc="Relevant policy snippets to ground the response. Can be empty string."
    )
    response: str = dspy.OutputField(
        desc="Final response to send to the customer — polite, clear, under 100 words"
    )


print("Signatures defined: ClassifyIntent, ExtractEntities, GenerateResponse")

Signatures defined: ClassifyIntent, ExtractEntities, GenerateResponse


## Section 3 — Agent Module

A **Module** chains signatures into a pipeline.

- `dspy.Predict` — direct answer, one LLM call
- `dspy.ChainOfThought` — adds a reasoning step before the answer (better for nuanced generation)

The `forward()` method is the pipeline: classify → extract → respond.

In [4]:
class CustomerCareAgent(dspy.Module):
    def __init__(self):
        self.classify = dspy.Predict(ClassifyIntent)
        self.extract  = dspy.Predict(ExtractEntities)
        self.respond  = dspy.ChainOfThought(GenerateResponse)  # CoT for better responses

    def forward(self, user_message: str, policy_context: str = "") -> dspy.Prediction:
        # Step 1 — what does the customer want?
        intent_result = self.classify(user_message=user_message)

        # Step 2 — what data did they mention?
        entity_result = self.extract(user_message=user_message)

        # Step 3 — pack entities as JSON for the response generator
        entities_json = json.dumps({
            "order_id": entity_result.order_id,
            "email":    entity_result.customer_email,
            "issue":    entity_result.issue_summary,
        })

        # Step 4 — generate a grounded response
        final = self.respond(
            user_message=user_message,
            intent=intent_result.intent,
            entities=entities_json,
            policy_context=policy_context,
        )

        return dspy.Prediction(
            intent=intent_result.intent,
            confidence=intent_result.confidence,
            entities=entities_json,
            response=final.response,
        )


print("CustomerCareAgent defined")
print("Pipeline: ClassifyIntent → ExtractEntities → GenerateResponse (CoT)")

CustomerCareAgent defined
Pipeline: ClassifyIntent → ExtractEntities → GenerateResponse (CoT)


## Section 4 — Trainset

31 labeled examples across 5 intents. Each example only needs a `user_message` and `intent` label.

DSPy bootstraps entity extraction and response quality from these automatically.
The 80/20 split gives us a **trainset** (for optimization) and a **devset** (for evaluation).

In [5]:
RAW_EXAMPLES = [
    # track_order
    ("Where is my order #45231? It's been 10 days.",                     "track_order"),
    ("Can you tell me the status of order #99102?",                      "track_order"),
    ("My package hasn't arrived yet, order number is #77543.",           "track_order"),
    ("I placed an order last week, still no update on delivery.",        "track_order"),
    ("Track my shipment please, order #33210.",                          "track_order"),
    ("When will order #12001 be delivered?",                             "track_order"),
    ("I haven't received any tracking info for my order.",               "track_order"),
    # cancel_order
    ("I want to cancel my order immediately.",                           "cancel_order"),
    ("Please cancel order #88421 before it ships.",                      "cancel_order"),
    ("I changed my mind, can you cancel my recent purchase?",            "cancel_order"),
    ("Cancel everything I ordered yesterday.",                           "cancel_order"),
    ("I need to stop my order from being processed.",                    "cancel_order"),
    ("How do I cancel my subscription?",                                 "cancel_order"),
    # refund_request
    ("I want a full refund for order #55210.",                           "refund_request"),
    ("The product I received was broken, I need my money back.",         "refund_request"),
    ("I was charged twice for the same order, please refund one.",       "refund_request"),
    ("My item never arrived and I want a refund.",                       "refund_request"),
    ("I returned the product last week, where is my refund?",            "refund_request"),
    ("I was overcharged, I'd like a refund for the difference.",         "refund_request"),
    # escalate
    ("This is the third time I'm contacting you and nobody helps me!",   "escalate"),
    ("I've been waiting 3 weeks, this is completely unacceptable.",      "escalate"),
    ("I want to speak to a manager right now.",                          "escalate"),
    ("Your customer service is terrible, I'm going to file a complaint.","escalate"),
    ("I have escalated this twice already with no resolution.",          "escalate"),
    ("Nobody is responding to my emails, I need a supervisor.",          "escalate"),
    # general_query
    ("What are your business hours?",                                    "general_query"),
    ("Do you ship internationally?",                                     "general_query"),
    ("What is your return policy?",                                      "general_query"),
    ("How long does standard shipping take?",                            "general_query"),
    ("Can I change my delivery address?",                                "general_query"),
    ("Do you have a loyalty rewards program?",                           "general_query"),
]


def build_trainset():
    return [
        dspy.Example(user_message=msg, intent=intent).with_inputs("user_message")
        for msg, intent in RAW_EXAMPLES
    ]


def get_splits():
    examples = build_trainset()
    split = int(len(examples) * TRAIN_SPLIT)
    return examples[:split], examples[split:]


trainset, devset = get_splits()

print(f"Total examples : {len(RAW_EXAMPLES)}")
print(f"Trainset       : {len(trainset)} examples (used by optimizer)")
print(f"Devset         : {len(devset)} examples (used for evaluation)")
print(f"\nSample example : {trainset[0]}")

Total examples : 31
Trainset       : 24 examples (used by optimizer)
Devset         : 7 examples (used for evaluation)

Sample example : Example({'user_message': "Where is my order #45231? It's been 10 days.", 'intent': 'track_order'}) (input_keys={'user_message'})


## Section 5 — Metrics

A **metric** is a Python function `(example, prediction, trace=None) → bool | float`.

The optimizer maximizes whatever this returns — so defining it well is the most important design decision.

Two metrics:
- `intent_accuracy` — exact match. Start here.
- `combined_metric` — 60% intent + 40% LLM-judge response quality. Use once intent is stable (>85%).

In [6]:
# ── Primary metric: exact intent match ────────────────────────────────────────
def intent_accuracy(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
    """
    Returns True if predicted intent matches the ground truth label.
    Simple, fast, no extra LLM calls. Start with this.
    """
    return (
        example.intent.strip().lower()
        == prediction.intent.strip().lower()
    )


# ── LLM-as-judge signature ─────────────────────────────────────────────────────
class JudgeResponseQuality(dspy.Signature):
    """Judge whether the customer care response is helpful, polite, and relevant."""
    user_message: str = dspy.InputField()
    response: str     = dspy.InputField()
    score: float      = dspy.OutputField(
        desc="Float between 0.0 (poor) and 1.0 (excellent)"
    )

_judge = dspy.Predict(JudgeResponseQuality)


# ── Combined metric ───────────────────────────────────────────────────────────
def combined_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> float:
    """
    60% intent accuracy (objective) + 40% LLM-judge response quality (subjective).
    Each call makes one extra LLM call for the judge.
    Use after intent accuracy is consistently above ~85%.
    """
    intent_score = float(intent_accuracy(example, prediction))
    try:
        quality = _judge(
            user_message=example.user_message,
            response=prediction.response,
        )
        quality_score = float(quality.score)
    except Exception:
        quality_score = 0.5
    return (intent_score * 0.6) + (quality_score * 0.4)


print("Metrics defined: intent_accuracy, combined_metric")
print("Active metric for optimization: intent_accuracy")

Metrics defined: intent_accuracy, combined_metric
Active metric for optimization: intent_accuracy


## Section 6 — Baseline Evaluation

Run the agent **before any optimization** to get a baseline accuracy score.
This uses only the docstring + field names — no few-shot examples, no optimized instructions.

In [7]:
def evaluate(agent, devset, label="") -> float:
    """Run agent on devset and print accuracy."""
    correct = 0
    results = []
    for ex in devset:
        try:
            pred = agent(user_message=ex.user_message)
            passed = intent_accuracy(ex, pred)
            if passed:
                correct += 1
            results.append({
                "message": ex.user_message[:50],
                "expected": ex.intent,
                "predicted": pred.intent,
                "pass": "✓" if passed else "✗"
            })
        except Exception as e:
            print(f"  [!] Error: {e}")
            results.append({"message": ex.user_message[:50], "expected": ex.intent, "predicted": "ERROR", "pass": "✗"})

    accuracy = correct / len(devset) if devset else 0.0
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    for r in results:
        print(f"  {r['pass']}  [{r['expected']:>15}]  {r['message']}")
    print(f"{'─'*55}")
    print(f"  Accuracy: {correct}/{len(devset)} = {accuracy:.1%}")
    print(f"{'='*55}\n")
    return accuracy

In [8]:
# Run baseline — no optimization, just raw signatures
baseline_agent = CustomerCareAgent()
baseline_accuracy = evaluate(baseline_agent, devset, label="BASELINE (no optimization)")


  BASELINE (no optimization)
  ✓  [       escalate]  Nobody is responding to my emails, I need a superv
  ✓  [  general_query]  What are your business hours?
  ✓  [  general_query]  Do you ship internationally?
  ✓  [  general_query]  What is your return policy?
  ✓  [  general_query]  How long does standard shipping take?
  ✓  [  general_query]  Can I change my delivery address?
  ✓  [  general_query]  Do you have a loyalty rewards program?
───────────────────────────────────────────────────────
  Accuracy: 7/7 = 100.0%



## Section 7 — Optimization

The optimizer runs **once** and produces a compiled prompt saved to disk.

**Toggle at the top of this notebook** (`OPTIMIZER` variable):

| Optimizer | What it does | Cost |
|---|---|---|
| `bootstrap` | Finds best few-shot demos from trainset | Low — good for a quick demo |
| `miprov2` | Teacher rewrites instructions + Bayesian search for best demo combo | High — best results |

### MIPROv2 internals (4 stages):
1. **Bootstrap demos** — runs trainset, collects correct traces
2. **Propose instructions** — teacher LLM writes N instruction variants
3. **Bayesian search** — samples (instruction, demo) combos, scores, learns, picks smarter next trial
4. **Compile best** — saves highest-scoring config to `optimized/`

In [9]:
import logging

# ── Trace level for the optimizer ──────────────────────────────────────────────
#   0 = quiet    — only final results
#   1 = progress — trial scores, best-so-far, proposed instructions  (recommended)
#   2 = full     — everything above + every prompt/response DSPy sends to the LLM
TRACE_LEVEL = 1


def _apply_trace_level(level: int):
    """Map our 0/1/2 dial onto DSPy's verbose flag + Python logging level."""
    log_level = {0: logging.WARNING, 1: logging.INFO, 2: logging.DEBUG}.get(level, logging.INFO)
    logging.getLogger("dspy").setLevel(log_level)
    # make sure logs actually show up in the notebook
    logging.basicConfig(level=log_level, format="%(message)s")
    return level >= 1   # verbose=True for level 1 and 2


def run_miprov2(trainset, devset):
    """
    MIPROv2 — teacher/student split + Bayesian search.
    Teacher (qwen2.5:32b) proposes instructions.
    Student (qwen2.5:14b) gets optimized and is what you deploy.

    Configured for a FAST learning run (a few trials, not full search).
    For real quality, set auto="medium"/"heavy" and drop the manual counts.
    """
    verbose = _apply_trace_level(TRACE_LEVEL)

    print(f"  Teacher    : {TEACHER_MODEL_ID}")
    print(f"  Student    : {STUDENT_MODEL_ID}")
    print(f"  Mode       : fast learning (2 candidates × 3 trials)")
    print(f"  Trace level: {TRACE_LEVEL}  (verbose={verbose})")
    print(f"  Trainset   : {len(trainset)} examples | Devset: {len(devset)} examples")

    # Fast learning config: auto=None lets us hand-pick tiny trial counts.
    # (auto="light" still runs ~10 trials × several instruction candidates.)
    optimizer = MIPROv2(
        metric=intent_accuracy,
        auto=None,                                # disable auto so num_candidates/num_trials apply
        num_candidates=2,                         # only 2 instruction variants to propose
        max_bootstrapped_demos=2,
        max_labeled_demos=2,
        teacher_settings=dict(lm=TEACHER_MODEL),  # teacher_settings goes on the constructor
        verbose=verbose,                          # driven by TRACE_LEVEL above
    )
    compiled = optimizer.compile(
        student=CustomerCareAgent(),
        trainset=trainset,
        valset=devset,
        num_trials=3,                             # just 3 search trials
        minibatch=False,                          # devset is tiny — full eval is fine
        requires_permission_to_run=False,
    )

    # At trace level 2, dump the raw prompts/responses the optimizer just tried
    if TRACE_LEVEL >= 2:
        print("\n" + "=" * 70)
        print("LAST LLM CALLS DURING OPTIMIZATION (TRACE_LEVEL=2)")
        print("=" * 70)
        dspy.inspect_history(n=6)

    return compiled


def run_bootstrap(trainset, devset):
    """
    BootstrapFewShot — fast, no instruction rewriting.
    Finds the examples where the metric passed and injects them as few-shot demos.
    """
    _apply_trace_level(TRACE_LEVEL)

    print(f"  Model    : {STUDENT_MODEL_ID}")
    print(f"  Max demos: {BOOTSTRAP_MAX_DEMOS} per module")
    print(f"  Trainset : {len(trainset)} examples")

    optimizer = BootstrapFewShot(
        metric=intent_accuracy,
        max_bootstrapped_demos=BOOTSTRAP_MAX_DEMOS,
        max_labeled_demos=BOOTSTRAP_MAX_DEMOS,
    )
    return optimizer.compile(
        student=CustomerCareAgent(),
        trainset=trainset,
    )


# ── Run the selected optimizer ─────────────────────────────────────────────────
print(f"Running optimizer: {OPTIMIZER.upper()}\n")

if OPTIMIZER == "miprov2":
    optimized_agent = run_miprov2(trainset, devset)
    save_path = MIPRO_SAVE_PATH
else:
    optimized_agent = run_bootstrap(trainset, devset)
    save_path = BOOTSTRAP_SAVE_PATH

# Save compiled prompt to disk
optimized_agent.save(save_path)
print(f"\nCompiled prompt saved to: {save_path}")

2026/06/27 16:11:57 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.


2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=2 sets of demonstrations...
2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Running optimizer: MIPROV2

  Teacher    : ollama/qwen2.5:32b
  Student    : ollama/qwen2.5:14b
  Mode       : fast learning (2 candidates × 3 trials)
  Trace level: 1  (verbose=True)
  Trainset   : 24 examples | Devset: 7 examples
Bootstrapping set 1/2
Bootstrapping set 2/2
SOURCE CODE: ClassifyIntent(user_message -> intent, confidence
    instructions='Classify the customer support message into a single known intent.'
    user_message = Field(annotation=str required=True json_schema_extra={'desc': 'Raw message from the customer', '__dspy_field_type': 'input', 'prefix': 'User Message:'})
    intent = Field(annotation=str required=True json_schema_extra={'desc': 'Exactly one of: track_order, cancel_order, refund_request, escalate, general_query', '__dspy_field_type': 'output', 'prefix': 'Intent:'})
    confidence = Field(annotation=float required=True json_schema_extra={'desc': 'Confidence score between 0.0 and 1.0', '__dspy_field_type': 'output', 'prefix': 'Confidence:'})
)



Extract

2026/06/27 16:11:57 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=2 instructions...



Using a randomly generated configuration for our grounded proposer.
Selected tip: description


...


Predictor 0
i: Classify the customer support message into a single known intent.
p: Confidence:
Predictor 1
i: Extract structured entities from the customer message.
p: Issue Summary:
Predictor 2
i: Given a user message from a customer support interaction, classify the intent of the message into one of the predefined categories ('track_order', 'cancel_order', 'refund_request', 'escalate', or 'general_query'). Extract structured entities such as order IDs and customer emails if mentioned. Utilize these extracted elements along with any provided policy context to generate a polite, clear, and concise response that addresses the customer's concern effectively within 100 words.
p: Response:


Average Metric: 7.00 / 7 (100.0%): 100%|██████████| 7/7 [06:06<00:00, 52.29s/it] 

2026/06/27 16:24:23 INFO dspy.evaluate.evaluate: Average Metric: 7 / 7 (100.0%)
2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 1', 'Predictor 2: Instruction 1', 'Predictor 2: Few-Shot Set 0'].
2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [100.0, 85.71, 100.0]
2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 4 / 3 =====
2026/06/27 16:24:23 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: Classify the customer support message into a single known intent.
p: Confidence:
Predictor 1
i: You are a customer support assistant designed to extract important information from user messages. Your task is to identify key details such as order IDs and email addresses, along with summarizing the main issue or query presented by each user. 

Given an unstructured customer message, please return:
- The `order_id` if mentioned (e.g., #12345), otherwise leave this field empty.
- The `customer_email` if provided in the message, otherwise provide an empty string for this field.
- A one-sentence summary of the issue or query presented by the user.

Example User Message: "Hi, I recently placed order #ABCD123 and haven't received it yet. Can you please check its status?"
Output Fields:
Order Id: ABCD123
Customer Email: 
Issue Summary: The customer has not received their recent order and wants to know the current status.
p: Issue Summary:
Predictor 2
i: Generate a helpful, polit

2026/06/27 16:29:41 INFO dspy.evaluate.evaluate: Average Metric: 7 / 7 (100.0%)
2026/06/27 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 1', 'Predictor 2: Instruction 0', 'Predictor 2: Few-Shot Set 1'].
2026/06/27 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [100.0, 85.71, 100.0, 100.0]
2026/06/27 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/06/27 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/06/27 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!




Compiled prompt saved to: optimized/miprov2_agent.json


## Section 8 — Before vs After Comparison

Run the **optimized agent** on the same devset and compare against the baseline.

In [10]:
optimized_accuracy = evaluate(optimized_agent, devset, label=f"OPTIMIZED ({OPTIMIZER.upper()})")

# Summary
improvement = optimized_accuracy - baseline_accuracy
print(f"  Baseline accuracy  : {baseline_accuracy:.1%}")
print(f"  Optimized accuracy : {optimized_accuracy:.1%}")
print(f"  Improvement        : +{improvement:.1%}" if improvement >= 0 else f"  Change: {improvement:.1%}")


  OPTIMIZED (MIPROV2)
  ✓  [       escalate]  Nobody is responding to my emails, I need a superv
  ✓  [  general_query]  What are your business hours?
  ✓  [  general_query]  Do you ship internationally?
  ✓  [  general_query]  What is your return policy?
  ✓  [  general_query]  How long does standard shipping take?
  ✓  [  general_query]  Can I change my delivery address?
  ✓  [  general_query]  Do you have a loyalty rewards program?
───────────────────────────────────────────────────────
  Accuracy: 7/7 = 100.0%

  Baseline accuracy  : 100.0%
  Optimized accuracy : 100.0%
  Improvement        : +0.0%


### What did the optimizer actually change?

Inspect the compiled prompt — see what instructions and few-shot examples were selected.

In [12]:
# Inspect the optimized ClassifyIntent signature
print("Optimized ClassifyIntent prompt:")
print("─" * 55)
print(optimized_agent.classify.signature)
print("\nFew-shot demos injected:")
print("─" * 55)
demos = getattr(optimized_agent.classify, 'demos', [])
if demos:
    for i, demo in enumerate(demos, 1):
        print(f"  Demo {i}: {demo}")
else:
    print("  (no demos — run with miprov2 or bootstrap to see injected examples)")

Optimized ClassifyIntent prompt:
───────────────────────────────────────────────────────
ClassifyIntent(user_message -> intent, confidence
    instructions='Classify the customer support message into a single known intent.'
    user_message = Field(annotation=str required=True json_schema_extra={'desc': 'Raw message from the customer', '__dspy_field_type': 'input', 'prefix': 'User Message:'})
    intent = Field(annotation=str required=True json_schema_extra={'desc': 'Exactly one of: track_order, cancel_order, refund_request, escalate, general_query', '__dspy_field_type': 'output', 'prefix': 'Intent:'})
    confidence = Field(annotation=float required=True json_schema_extra={'desc': 'Confidence score between 0.0 and 1.0', '__dspy_field_type': 'output', 'prefix': 'Confidence:'})
)

Few-shot demos injected:
───────────────────────────────────────────────────────
  (no demos — run with miprov2 or bootstrap to see injected examples)


## Section 9 — Inference (Production)

Load the compiled agent from disk and run it on real customer messages.

This is exactly what production looks like:
- No optimizer running
- No training loop
- Just the static compiled prompt + one LLM call per request

The `predict_with_rag` function simulates a RAG lookup — in production, replace the `POLICY_SNIPPETS` dict with a real vector DB call.

In [13]:
# Simulated policy context (replace with real RAG in production)
POLICY_SNIPPETS = {
    "track_order":    "Tracking updates are available within 24 hours of dispatch.",
    "cancel_order":   "Orders can be cancelled within 1 hour of placement. After that, contact support.",
    "refund_request": "Refunds are processed within 5-7 business days to the original payment method.",
    "escalate":       "Escalated cases are reviewed by a senior agent within 2 business hours.",
    "general_query":  "Standard shipping takes 3-5 business days. Express shipping is 1-2 business days.",
}


def load_agent(path: str) -> CustomerCareAgent:
    """Load the compiled agent from disk. No optimizer needed."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' not found. Run the optimization cell first."
        )
    dspy.configure(lm=STUDENT_MODEL)
    agent = CustomerCareAgent()
    agent.load(path)
    return agent


def predict_with_rag(agent: CustomerCareAgent, user_message: str) -> dspy.Prediction:
    """
    Two-pass inference:
      Pass 1 — classify to get intent
      Pass 2 — full response with policy context injected
    In production: replace POLICY_SNIPPETS.get() with a vector DB call.
    """
    quick  = agent(user_message=user_message, policy_context="")
    policy = POLICY_SNIPPETS.get(quick.intent, "")
    return agent(user_message=user_message, policy_context=policy)


def print_result(user_message: str, result: dspy.Prediction) -> None:
    print(f"  User     : {user_message}")
    print(f"  Intent   : {result.intent}  (confidence: {result.confidence})")
    print(f"  Entities : {result.entities}")
    print(f"  Response : {result.response}")
    print("  " + "─" * 58)


# Load the compiled agent
compiled_path = MIPRO_SAVE_PATH if OPTIMIZER == "miprov2" else BOOTSTRAP_SAVE_PATH
prod_agent = load_agent(compiled_path)
print(f"Loaded compiled agent from: {compiled_path}")
print("Ready for inference.")

Loaded compiled agent from: optimized/miprov2_agent.json
Ready for inference.


In [14]:
# Run the agent on test cases
test_cases = [
    "My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.",
    "I want to cancel my order right now.",
    "I've called 5 times and nobody helps me. This is ridiculous.",
    "Do you ship to Singapore?",
    "Where is my package? Order #99102.",
]

print("=" * 60)
print("LIVE INFERENCE — optimized agent")
print("=" * 60 + "\n")

for msg in test_cases:
    result = predict_with_rag(prod_agent, msg)
    print_result(msg, result)

LIVE INFERENCE — optimized agent

  User     : My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.
  Intent   : refund_request  (confidence: 0.95)
  Entities : {"order_id": "#45231", "email": "(empty string)", "issue": "The customer's order #45231 has not arrived after 2 weeks, and they are requesting a refund."}
  Response : I understand your concern about the delayed shipment of your order #45231. We will initiate a full refund for you and it should be processed within 5-7 business days. Please allow some time for this process, and if you have any further questions or issues, feel free to reach out.
  ──────────────────────────────────────────────────────────
  User     : I want to cancel my order right now.
  Intent   : cancel_order  (confidence: 1.0)
  Entities : {"order_id": "", "email": "", "issue": "The customer wants to cancel their order."}
  Response : Thank you for your message. To help you better, could you please provide me with the order ID and your ema

## Section 9b — Trace Every Step

See the **actual prompts and completions** DSPy sends to the model for each step
of the pipeline (classify → extract → respond).

Two ways:
1. `dspy.inspect_history(n=N)` — raw prompt + response of the last N LLM calls.
2. `dspy.settings.trace` — the structured list of `(predictor, inputs, outputs)` tuples,
   captured inside a `dspy.context(trace=[])` block.

In [15]:
# ── Method 1: raw prompts + completions (inspect_history) ──────────────────────
# Run one message through the agent — this fires 3 LLM calls:
#   classify → extract → respond (CoT)
trace_msg = "My order #45231 hasn't arrived and it's been 2 weeks. I want a refund."
_ = prod_agent(user_message=trace_msg, policy_context="")

print("=" * 70)
print("RAW LLM HISTORY — last 3 calls (classify, extract, respond)")
print("=" * 70)
# n=3 → the 3 steps of the pipeline, each with its full prompt + model output
dspy.inspect_history(n=3)

RAW LLM HISTORY — last 3 calls (classify, extract, respond)




[2026-06-27T16:42:03.807133]

System message:

Your input fields are:
1. `user_message` (str): Raw message from the customer
Your output fields are:
1. `intent` (str): Exactly one of: track_order, cancel_order, refund_request, escalate, general_query
2. `confidence` (float): Confidence score between 0.0 and 1.0
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## user_message ## ]]
{user_message}

[[ ## intent ## ]]
{intent}

[[ ## confidence ## ]]
{confidence}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the customer support message into a single known intent.


User message:

[[ ## user_message ## ]]
My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.

Respond with the corresponding output fields, starting with the field `[[ ## intent ## ]]`,

In [16]:
# ── Method 2: structured trace (predictor → inputs → outputs) ──────────────────
# dspy.context(trace=[]) collects each predictor call as a tuple while the
# agent runs. Cleaner than parsing raw prompts when you just want step I/O.
STEP_NAMES = ["1. ClassifyIntent", "2. ExtractEntities", "3. GenerateResponse"]

with dspy.context(trace=[]):
    prod_agent(user_message=trace_msg, policy_context="")
    steps = list(dspy.settings.trace)   # [(predictor, inputs_dict, prediction), ...]

print("=" * 70)
print("STRUCTURED TRACE — each pipeline step")
print("=" * 70)
for i, (predictor, inputs, outputs) in enumerate(steps):
    name = STEP_NAMES[i] if i < len(STEP_NAMES) else f"step {i+1}"
    print(f"\n▶ {name}  ({predictor.signature.__name__})")
    print("  ── inputs ──")
    for k, v in inputs.items():
        print(f"    {k}: {str(v)[:120]}")
    print("  ── outputs ──")
    for k, v in outputs.items():
        print(f"    {k}: {str(v)[:120]}")

STRUCTURED TRACE — each pipeline step

▶ 1. ClassifyIntent  (StringSignature)
  ── inputs ──
    user_message: My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.
  ── outputs ──
    intent: refund_request
    confidence: 0.95

▶ 2. ExtractEntities  (StringSignature)
  ── inputs ──
    user_message: My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.
  ── outputs ──
    order_id: #45231
    customer_email: (empty string)
    issue_summary: The customer's order #45231 has not arrived after 2 weeks, and they are requesting a refund.

▶ 3. GenerateResponse  (StringSignature)
  ── inputs ──
    user_message: My order #45231 hasn't arrived and it's been 2 weeks. I want a refund.
    intent: refund_request
    entities: {"order_id": "#45231", "email": "(empty string)", "issue": "The customer's order #45231 has not arrived after 2 weeks, a
    policy_context: 
  ── outputs ──
    reasoning: The customer is requesting a refund due to their order not arrivi

## Section 10 — Try Your Own Message

Type any customer message and run the cell to see what the agent returns.

In [17]:
# ── Try your own message here ─────────────────────────────────────────────────
YOUR_MESSAGE = "I was charged twice for the same item last Tuesday."
# ─────────────────────────────────────────────────────────────────────────────

result = predict_with_rag(prod_agent, YOUR_MESSAGE)
print_result(YOUR_MESSAGE, result)

  User     : I was charged twice for the same item last Tuesday.
  Intent   : refund_request  (confidence: 0.95)
  Entities : {"order_id": "{empty}", "email": "{empty}", "issue": "The user was charged twice for the same item last Tuesday."}
  Response : I'm sorry to hear that you've been charged twice for the same item. To help us process your request efficiently, could you please provide me with your order ID and email address? Once we have these details, I'll be able to look into this issue right away.
  ──────────────────────────────────────────────────────────


## Summary

| Concept | What you write | What DSPy generates |
|---|---|---|
| **Signature docstring** | Task instruction | Top line of the prompt |
| **InputField / OutputField** | Field names + desc | Format block + output slots |
| **Metric** | A Python scoring function | The goal the optimizer maximizes |
| **Optimizer** | `OPTIMIZER = "miprov2"` | Best instruction text + few-shot examples |
| **Compiled prompt** | Saved to `optimized/*.json` | Loaded at runtime, no optimizer needed |

---

### Next steps

- **Add more examples**: extend `RAW_EXAMPLES` and re-run Section 7
- **Add RAG**: replace `POLICY_SNIPPETS.get()` in `predict_with_rag` with a ChromaDB/pgvector call
- **Better metric**: switch to `combined_metric` once intent accuracy is stable above 85%
- **Switch to MIPROv2 `medium`**: change `MIPRO_AUTO = "medium"` in Section 1 for better results